# LightRAG Evaluation Pipeline — Google Colab

This notebook runs the full eval pipeline (ingest → query → judge → report) using
**Google Gemini** as the LLM and embedding backend — **completely free**, no GPU needed.

### What you need
1. A **free Gemini API key** → [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)
2. Your **document corpus** (`.txt` / `.md` files)
3. Your **multihop questions** in the JSON format shown in Cell 4

### Runtime
- **CPU runtime is fine** (Gemini inference happens in Google's cloud)
- Go to `Runtime → Change runtime type → CPU` to save GPU quota

---
## Step 1 — Clone the repo and install dependencies

In [ ]:
import os

# Clone the repo (skip if already cloned)
if not os.path.exists('LightRAG'):
    !git clone https://github.com/anasmostafa23/LightRAG.git

os.chdir('LightRAG')
print('Working directory:', os.getcwd())

# Install LightRAG + Gemini + OpenAI client (needed for judge)
!pip install -q -e ".[offline-llm]" openai nest_asyncio

---
## Step 2 — Configuration

**Edit this cell** before running anything else.

In [ ]:
import os

# ── API Key ────────────────────────────────────────────────────────────────
# Option A: paste it directly (not recommended for shared notebooks)
GEMINI_API_KEY = ""   # ← paste your key here

# Option B: use Colab Secrets (recommended)
# 1. Click the 🔑 key icon in the left sidebar
# 2. Add a secret named GEMINI_API_KEY with your key value
# 3. Uncomment the two lines below and comment out Option A above
# from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# ── Models ─────────────────────────────────────────────────────────────────
# gemini-2.0-flash  → best free option (fast, 1M tokens/day)
# gemini-1.5-flash  → also free, slightly older
LLM_MODEL   = "gemini-2.0-flash"
EMBED_MODEL = "models/text-embedding-004"   # 768-dim, free
JUDGE_MODEL = "gemini-2.0-flash"             # same model for judging

# ── Query modes to evaluate ────────────────────────────────────────────────
# Remove any you don't want; "mix" is the most complete
MODES = ["naive", "local", "global", "hybrid", "mix"]

# ── Paths ──────────────────────────────────────────────────────────────────
WORKING_DIR     = "/content/LightRAG/eval/rag_storage"
CORPUS_DIR      = "/content/LightRAG/eval/corpus"
QUESTIONS_FILE  = "/content/LightRAG/eval/questions.json"
RESULTS_DIR     = "/content/LightRAG/eval/results"

# ── Pipeline flags ─────────────────────────────────────────────────────────
SKIP_INGESTION = False   # True = reuse an existing index (faster re-runs)
FORCE_REINDEX  = False   # True = wipe + rebuild from scratch

print('Config OK')
assert GEMINI_API_KEY, 'Set your GEMINI_API_KEY above!'

---
## Step 3 — Upload your documents

Run the cell, then use the file picker to select your `.txt` / `.md` files.
They will be saved to `eval/corpus/`.

In [ ]:
import os
from google.colab import files

os.makedirs(CORPUS_DIR, exist_ok=True)

print(f'Upload your .txt or .md documents (they will be saved to {CORPUS_DIR})')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = os.path.join(CORPUS_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest} ({len(content):,} bytes)')

doc_files = [f for f in os.listdir(CORPUS_DIR) if f.endswith(('.txt', '.md'))]
print(f'\nCorpus now has {len(doc_files)} file(s): {doc_files}')

---
## Step 4 — Define your multihop questions

Edit the list below. Each question needs:
- `id` — unique string identifier
- `question` — the question text
- `ground_truth` — the complete correct answer (covering all hops)
- `hop_count` — how many documents/facts must be chained (informational)

In [ ]:
import json, os

questions = [
    {
        "id": "q1",
        "hop_count": 2,
        "question": "REPLACE: Your first 2-hop question here.",
        "ground_truth": "REPLACE: Complete answer covering both reasoning hops.",
        "metadata": {"difficulty": "medium"}
    },
    {
        "id": "q2",
        "hop_count": 3,
        "question": "REPLACE: Your second 3-hop question here.",
        "ground_truth": "REPLACE: Complete answer covering all three reasoning hops.",
        "metadata": {"difficulty": "hard"}
    },
]

os.makedirs(os.path.dirname(QUESTIONS_FILE), exist_ok=True)
with open(QUESTIONS_FILE, 'w') as f:
    json.dump({'questions': questions}, f, indent=2)

print(f'Saved {len(questions)} question(s) to {QUESTIONS_FILE}')

---
## Step 5 — Run the evaluation pipeline

This does everything in sequence:
1. Ingest documents → build knowledge graph
2. Query each question across all selected modes
3. Score each answer with the judge model
4. Print summary table + save CSV/JSON results

In [ ]:
import sys, asyncio, nest_asyncio
nest_asyncio.apply()  # allow asyncio.run() inside Colab's event loop

# Ensure the repo root is on the path
REPO_ROOT = '/content/LightRAG'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from eval.config import EvalConfig
from eval.pipeline import run

cfg = EvalConfig(
    backend          = 'gemini',
    gemini_api_key   = GEMINI_API_KEY,
    gemini_llm_model = LLM_MODEL,
    gemini_embed_model = EMBED_MODEL,
    gemini_embed_dim = 768,
    judge_model      = JUDGE_MODEL,
    modes            = MODES,
    working_dir      = WORKING_DIR,
    corpus_dir       = CORPUS_DIR,
    questions_file   = QUESTIONS_FILE,
    results_dir      = RESULTS_DIR,
    skip_ingestion   = SKIP_INGESTION,
    force_reindex    = FORCE_REINDEX,
)

asyncio.run(run(cfg))

---
## Step 6 — Explore results

In [ ]:
import pandas as pd
import glob, os

csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
if not csv_files:
    print('No results yet — run Step 5 first.')
else:
    latest = csv_files[-1]
    print(f'Showing: {latest}\n')
    df = pd.read_csv(latest)

    # Summary per mode
    score_cols = ['faithfulness', 'answer_relevance', 'completeness', 'coherence', 'overall']
    summary = df.groupby('mode')[score_cols].mean().round(2)
    print('=== Mean scores per mode (1–5 scale) ===')
    display(summary)

    print('\n=== Full results ===')
    display(df[['question_id', 'hop_count', 'mode', 'overall', 'latency_s', 'answer']].head(30))

In [ ]:
# Optional: download the latest CSV to your machine
from google.colab import files as colab_files
import glob, os

csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
if csv_files:
    colab_files.download(csv_files[-1])
else:
    print('No CSV results to download yet.')

---
## Tips

**Re-run queries without re-indexing** (saves time after first run):
```python
SKIP_INGESTION = True
```
then re-run Step 5.

**Only score specific modes**:
```python
MODES = ['hybrid', 'mix']
```

**Gemini free tier limits** (as of 2025):
- `gemini-2.0-flash`: 1,500 requests/day, 1M tokens/day
- `text-embedding-004`: 1,500 requests/day
- If you hit limits, wait until midnight Pacific time or upgrade to paid tier.

**Scoring dimensions** (1–5 scale):
| Metric | Measures |
|---|---|
| faithfulness | No hallucinations — claims match the ground truth |
| answer_relevance | Directly answers the question asked |
| completeness | All multihop reasoning steps are covered |
| coherence | Logically structured, easy to follow |
| overall | Mean of the four above |